In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score

import torchvision.transforms as transforms
import torchvision.models as models
 
# ─────────────────────────────────────────────
# Path Configuration (edit this)
# ─────────────────────────────────────────────
# Path to cleaned CSV
CSV_PATH = "/Users/emilymoore/Downloads/DS 4002 Project 3/cleaned_breast_lesions.csv"

# Folder where images are stored
IMAGE_DIR = "/Users/emilymoore/Downloads/DS 4002 Project 3/Breast Lesions USG Images/"

# Folder where masks are stored
MASK_DIR = "/Users/emilymoore/Downloads/DS 4002 Project 3/Breast Lesions USG Masks/"
 
# ─────────────────────────────────────────────
# Load Cleaned Data
# ─────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

# Drop rows with missing filenames
df = df.dropna(subset=["Image_filename", "Mask_tumor_filename"])

# Ensure filenames are strings
df["Image_filename"] = df["Image_filename"].astype(str)
df["Mask_tumor_filename"] = df["Mask_tumor_filename"].astype(str)

# Optional: strip whitespace
df["Image_filename"] = df["Image_filename"].str.strip()
df["Mask_tumor_filename"] = df["Mask_tumor_filename"].str.strip()

# Encode classification labels
label_map = {"benign": 0, "malignant": 1, "normal": 2}
df["Classification"] = df["Classification"].str.lower().map(label_map)

# Drop missing labels
df = df.dropna(subset=["Classification"])

# ─────────────────────────────────────────────
# Dataset Class
# ─────────────────────────────────────────────
class BreastDataset(Dataset):
    def __init__(self, df, image_dir, mask_dir, transform=None, scaler=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.meta_cols = ["Age", "BIRADS"]

        if scaler is not None:
            self.meta = scaler.transform(df[self.meta_cols].fillna(0))
        else:
            self.meta = df[self.meta_cols].fillna(0).values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = row["Image_filename"]
        mask_name = row["Mask_tumor_filename"]

    # Safety check
        if not isinstance(img_name, str) or not isinstance(mask_name, str):
            raise ValueError(f"Invalid filename at index {idx}")

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        metadata = torch.tensor(self.meta[idx], dtype=torch.float32)
        label = torch.tensor(int(row["Classification"]), dtype=torch.long)

        return image, mask, metadata, label
 
 
# ─────────────────────────────────────────────
# Transforms
# ─────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ─────────────────────────────────────────────
# Train/test Split
# ─────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Normalize metadata
scaler = StandardScaler()
scaler.fit(train_df[["Age", "BIRADS"]].fillna(0))

train_dataset = BreastDataset(train_df, IMAGE_DIR, MASK_DIR, transform, scaler)
test_dataset  = BreastDataset(test_df,  IMAGE_DIR, MASK_DIR, transform, scaler)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ─────────────────────────────────────────────
# Models
# ─────────────────────────────────────────────
# Image only model
class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Linear(512, 3)

    def forward(self, x):
        return self.backbone(x)

# Image and Metadata Model
class MultiModalModel(nn.Module):
    def __init__(self, meta_dim=2):
        super().__init__()

        self.cnn = models.resnet18(weights="IMAGENET1K_V1")
        self.cnn.fc = nn.Identity()

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, 16),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 + 16, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, image, meta):
        img_feat = self.cnn(image)
        meta_feat = self.meta_net(meta)

        combined = torch.cat([img_feat, meta_feat], dim=1)
        return self.classifier(combined)

# ─────────────────────────────────────────────
# Train Function
# ─────────────────────────────────────────────
def train_model(model, train_loader, use_metadata=False, epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, masks, meta, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            meta = meta.to(device)

            if use_metadata:
                outputs = model(images, meta)
            else:
                outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    return model
 
# ─────────────────────────────────────────────
# Evaluation Function
# ─────────────────────────────────────────────
def evaluate_model(model, test_loader, use_metadata=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, masks, meta, labels in test_loader:
            images = images.to(device)
            meta = meta.to(device)

            if use_metadata:
                outputs = model(images, meta)
            else:
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="weighted")
    f1 = f1_score(all_labels, all_preds, average="weighted")

    return acc, prec, f1
 
# ─────────────────────────────────────────────
# Run Experiments
# ─────────────────────────────────────────────
# Image only
img_model = ImageClassifier()
img_model = train_model(img_model, train_loader, use_metadata=False)

acc, prec, f1 = evaluate_model(img_model, test_loader, use_metadata=False)

print("IMAGE ONLY")
print("Accuracy:", acc)
print("Precision:", prec)
print("F1:", f1)

# Image and Metadata Model
multi_model = MultiModalModel()
multi_model = train_model(multi_model, train_loader, use_metadata=True)

acc, prec, f1 = evaluate_model(multi_model, test_loader, use_metadata=True)

print("IMAGE + METADATA")
print("Accuracy:", acc)
print("Precision:", prec)
print("F1:", f1)